In [ ]:
import pandas as pd
import numpy as np

acs_need = pd.read_csv("Agg Data FINAL/Panhandle_Community_Need_Index.csv")
tracts = pd.read_csv("Agg Data FINAL/Panhandle_Census_Tracts.csv")
facilities = pd.read_excel("Agg Data FINAL/Panhandle_All_Facilities.xlsx")

print(acs_need.shape)
print(tracts.shape)
print(facilities.shape)

(27, 12)
(27, 4)
(17, 7)


In [2]:
print(tracts.columns)
print(facilities.columns)

Index(['County', 'Tract_GEOID', 'Latitude', 'Longitude'], dtype='object')
Index(['Facility_Name', 'County', 'City', 'Address', 'Longitude', 'Latitude',
       'Facility_Type'],
      dtype='object')


Note that these are straight line distances!

In [3]:
import numpy as np
from math import radians, sin, cos, sqrt, atan2

# calculate distance between two locations in miles
def calculate_distance(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    c = 2 * atan2(sqrt(a), sqrt(1-a))

    return 3958.8 * c

# find the nearest facility for each tract
nearest_distances = []

for i, tract in tracts.iterrows():
    distances = []

    for j, facility in facilities.iterrows():
        distance = calculate_distance(
            tract["Latitude"],
            tract["Longitude"],
            facility["Latitude"],
            facility["Longitude"])
        distances.append(distance)

    nearest_distances.append(min(distances))

tracts["Nearest_Facility_Miles"] = nearest_distances
display(tracts[["County", "Tract_GEOID", "Nearest_Facility_Miles"]])

,County,Tract_GEOID,Nearest_Facility_Miles
0,BANNER,31007954000,21.233244
1,BOX BUTTE,31013951100,5.860447
2,BOX BUTTE,31013951200,5.654517
3,BOX BUTTE,31013951300,5.731869
4,CHEYENNE,31033954800,6.236515
5,CHEYENNE,31033954900,2.576288
6,CHEYENNE,31033955000,0.565491
7,DAWES,31045950600,10.573570
8,DAWES,31045950700,2.324525
9,DEUEL,31049955400,13.819561


In [4]:
tracts.to_csv("Agg Data FINAL/Panhandle_Tract_Facility_Distances.csv",index=False)

In [6]:
# create matching tract descriptions using the GEOIDs
tracts["Tract_Number"] = (tracts["Tract_GEOID"].astype(str).str[-6:].astype(int) // 100)

tracts["Tract"] = ("Census Tract " + tracts["Tract_Number"].astype(str) + "; " + tracts["County"].str.title() + " County; Nebraska")

# combine community need and facility distances
project_data = acs_need.merge(tracts[["Tract", "Tract_GEOID", "Nearest_Facility_Miles"]],on="Tract",how="left")

print(project_data.shape)
print(project_data["Nearest_Facility_Miles"].isna().sum())

display(project_data[["Tract", "Community_Need_Index","Nearest_Facility_Miles"]])

(27, 14)
0


,Tract,Community_Need_Index,Nearest_Facility_Miles
0,Census Tract 9545; Kimball County; Nebraska,0.877977,3.124918
1,Census Tract 9537; Scotts Bluff County; Nebraska,0.821437,1.430032
2,Census Tract 9531; Scotts Bluff County; Nebraska,0.683019,3.248792
3,Census Tract 9535; Scotts Bluff County; Nebraska,0.650663,1.637908
4,Census Tract 9521; Garden County; Nebraska,0.534964,15.436605
5,Census Tract 9516; Sheridan County; Nebraska,0.422660,2.281271
6,Census Tract 9517; Sheridan County; Nebraska,0.182884,18.919640
7,Census Tract 9538; Scotts Bluff County; Nebraska,0.165542,2.255951
8,Census Tract 9554; Deuel County; Nebraska,0.086290,13.819561
9,Census Tract 9550; Cheyenne County; Nebraska,0.019821,0.565491


In [7]:
project_data.to_csv("Agg Data FINAL/Panhandle_Need_and_Access.csv",index=False)